In [0]:
%pip install langchain-text-splitters

In [0]:
from pyspark.sql import functions as F

docs_df = spark.table(
    "enterprise_ai.raw_data.product_docs"
)

display(docs_df)

In [0]:
docs_df.printSchema()

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# =========================
# ENTERPRISE CHUNKER
# =========================

splitter = RecursiveCharacterTextSplitter(

    chunk_size=1000,
    chunk_overlap=150,

    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]

)

# =========================
# CREATE CHUNKS
# =========================

chunk_rows = []

for row in docs_df.collect():

    text = row["product_doc"]

    if text is not None:

        chunks = splitter.split_text(text)

        for i, chunk in enumerate(chunks):

            chunk_rows.append({

                "product_name": row["product_name"],
                "chunk_id": i,
                "chunk_text": chunk,
                "company": "Infosys"

            })

# =========================
# CREATE DATAFRAME
# =========================

chunk_df = spark.createDataFrame(
    chunk_rows
)

display(chunk_df)

In [0]:
chunk_df = chunk_df.withColumn(

    "financial_year",

    F.when(
        F.col("product_name").contains("25"),
        "FY25"
    ).when(
        F.col("product_name").contains("24"),
        "FY24"
    ).when(
        F.col("product_name").contains("23"),
        "FY23"
    ).otherwise(
        "Unknown")
)

display(chunk_df)

In [0]:
chunk_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "enterprise_ai.raw_data.document_chunks"
    )

print("document_chunks created")

In [0]:
spark.table(
    "enterprise_ai.raw_data.document_chunks"
).show()

In [0]:
spark,sql("""
ALTER TABLE enterprise_ai.raw_data.document_chunks
SET TBLPROPERTIES(delta.enablechangeDataFeed=TRUE)
""")

In [0]:
from pyspark.ml.functions import predict_batch_udf
from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd
import numpy as np

# =========================
# LOAD CHUNK TABLE
# =========================

chunk_df = spark.table(
    "enterprise_ai.raw_data.document_chunks"
)

display(chunk_df)

In [0]:
pip install transformers


In [0]:
pip install sentence-transformers


In [0]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

texts = [

    row["chunk_text"]

    for row in chunk_df.collect()
]

embeddings = model.encode(texts)

chunk_pd = chunk_df.toPandas()

chunk_pd["embedding"] = embeddings.tolist()

embedded_df = spark.createDataFrame(
    chunk_pd
)

display(embedded_df)

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import ArrayType, FloatType

(embedded_df
    .select(
        "chunk_text", 
        "company",  
        col("embedding").cast(ArrayType(FloatType())).alias("embedding"), 
        
        "financial_year"
    )
    .write
    .mode("overwrite") 
    .option("overwriteSchema", "true") 
    .saveAsTable("enterprise_ai.raw_data.document_chunk_embeddings")
)


In [0]:
spark,sql("""
ALTER TABLE enterprise_ai.raw_data.document_chunk_embeddings 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
""")

